# 04 - B7/B8 Workspace Fix 验证 (v2)

第二轮修复，解决了上一轮遗漏的两个 bug：

**v2 修复内容**:
1. **Reviewer 重试时丢失 workspace 上下文** — 重试路径现在也走 `format_input()`，附带 workspace 文件列表
2. **重试间不重新同步 workspace** — 每次 attempt 前都调用 `_sync_workspace_files()`
3. **Coder prompt 更强势** — 明确 4 步操作顺序 + 禁止事项，`max_tool_rounds` 回调到 10

**第一轮修复（已包含）**: workspace 上下文注入、tool result 截断 8000 字符

**对比基线**:
- 原始 B7=606K tok / workspace 空
- v1 修复后 B7=895K tok / workspace 仍空（重试路径绕过 format_input 导致上下文丢失）

**预估耗时**: ~10-15 分钟（2 个任务）

In [ ]:
# Cell 1: 环境准备
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/CodeAgent-MCP"
os.chdir(PROJECT_DIR)
print(f"Working dir: {os.getcwd()}")

!pip install -q openai mcp pydantic pyyaml rich nest_asyncio

In [ ]:
# Cell 2: API Key
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")

with open("/tmp/.api_key", "w") as f:
    f.write(os.environ["OPENAI_API_KEY"])

print("API key configured")

In [ ]:
# Cell 3: B7 + B8 MCP 测试脚本
%%writefile /tmp/test_b7b8_fix.py
import asyncio
import json
import os
import shutil
import sys
import time
from datetime import datetime

os.chdir("/content/drive/MyDrive/CodeAgent-MCP")
sys.path.insert(0, ".")

with open("/tmp/.api_key") as f:
    os.environ["OPENAI_API_KEY"] = f.read().strip()

WORKSPACE = "/tmp/workspace_b7b8"
os.makedirs(WORKSPACE, exist_ok=True)
os.environ["FILE_SERVER_ROOT"] = WORKSPACE
os.environ["SHELL_SERVER_CWD"] = WORKSPACE
os.environ["GIT_SERVER_ROOT"] = WORKSPACE

from eval.run_eval import load_benchmark, run_single_task
from src.core.config import load_settings, load_agents_config, load_mcp_config

async def main():
    tasks = load_benchmark()
    selected = [t for t in tasks if t["task_id"] in ["B7", "B8"]]
    settings = load_settings()
    agents_config = load_agents_config()
    mcp_config = load_mcp_config()

    print("=" * 60)
    print("B7/B8 Workspace Fix Verification (MCP mode)")
    print("=" * 60)
    print(f"max_tool_rounds: {agents_config['coder'].get('max_tool_rounds', 5)}")
    print(f"max_tool_result_chars: {agents_config['coder'].get('max_tool_result_chars', 8000)}")
    print()

    results = []
    for task in selected:
        # 清理 workspace
        for item in os.listdir(WORKSPACE):
            p = os.path.join(WORKSPACE, item)
            if os.path.isdir(p):
                shutil.rmtree(p, ignore_errors=True)
            else:
                os.remove(p)

        print(f"\n{'='*60}")
        print(f"Task: {task['task_id']} - {task['name']} ({task['difficulty']})")
        print(f"{'='*60}")

        start = time.time()
        result = await run_single_task(
            task, settings, agents_config, mcp_config,
            provider="default", use_mcp=True,
        )
        elapsed = time.time() - start

        # 记录 workspace 文件
        ws_files = []
        for f in sorted(os.listdir(WORKSPACE)):
            fp = os.path.join(WORKSPACE, f)
            if os.path.isfile(fp):
                size = os.path.getsize(fp)
                ws_files.append({"name": f, "size": size})
                print(f"  FILE: {f} ({size} bytes)")

        result["workspace_files"] = [f["name"] for f in ws_files]
        result["workspace_details"] = ws_files
        results.append(result)

        ws_status = "OK" if ws_files else "EMPTY"
        print(f"\n  Score: {result.get('review_score', 'N/A')}")
        print(f"  Tokens: {result.get('total_tokens', '?'):,}")
        print(f"  Time: {elapsed:.1f}s")
        print(f"  Attempts: {result.get('attempts', '?')}")
        print(f"  Workspace: {ws_status} ({len(ws_files)} files)")

    # 汇总对比
    print(f"\n\n{'='*60}")
    print("SUMMARY: Before vs After Fix")
    print(f"{'='*60}")
    print(f"\n{'Task':<8} {'Before tokens':<16} {'After tokens':<16} {'Before ws':<12} {'After ws':<12} {'Score'}")
    print("-" * 75)

    before = {"B7": {"tokens": 606274, "ws": "EMPTY"}, "B8": {"tokens": 685091, "ws": "EMPTY"}}
    for r in results:
        tid = r["task_id"]
        b = before[tid]
        after_tok = r.get("total_tokens", 0)
        after_ws = "OK" if r.get("workspace_files") else "EMPTY"
        ratio = f"({after_tok/b['tokens']:.1%})" if b['tokens'] else ""
        print(f"{tid:<8} {b['tokens']:>12,}    {after_tok:>12,}  {ratio:>8}  {b['ws']:<12} {after_ws:<12} {r.get('review_score', 'N/A')}")

    # 保存 JSON
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output = {
        "experiment": "b7b8_fix_verification",
        "timestamp": timestamp,
        "fix_description": [
            "Coder prompt: prioritize file_write over exploration",
            "Workspace context injection via _sync_workspace_files",
            "Tool result truncation (max 8000 chars)",
            "max_tool_rounds 10->15",
        ],
        "before_baseline": before,
        "results": results,
    }
    os.makedirs("eval/results", exist_ok=True)
    path = f"eval/results/b7b8_fix_{timestamp}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    print(f"\nResults saved to: {path}")

asyncio.run(main())

In [ ]:
!python /tmp/test_b7b8_fix.py

## 预期结果

| 指标 | 原始 B7 | v1 B7 | v2 B7 | 原始 B8 | v1 B8 | v2 B8 |
|------|--------|-------|-------|--------|-------|-------|
| Workspace | EMPTY | EMPTY | **有文件** | EMPTY | pyproject.toml | **有文件** |
| Tokens | 606K | 895K | **下降** | 685K | 753K | **下降** |
| Score | 8.5 | 9.2 | >=8.5 | 9.0 | 9.0 | >=8.5 |

如果 workspace 仍为空，检查 Colab 输出中的日志：
- `[Orchestrator] Workspace files: [...]` — 确认 workspace sync 在工作
- `[Coder Agent] Round N: called [...]` — 观察 Coder 是否调用了 file_write

结果 JSON 已保存到 `eval/results/b7b8_fix_*.json`，下载到本地后让 Claude Code 分析。